# 61 — Prospective Shadow Cohorts

Registra únicamente batches cuyo `t0` ocurre después del inicio prospectivo congelado. No usa labels y no evalúa performance todavía.

El objetivo es demostrar operación longitudinal real: múltiples batches, temporalidad correcta y cero contaminación con holdouts consumidos.


In [1]:
from argos.common.paths import ProjectPaths
from argos.config.prospective import load_prospective_spec
from argos.config.production import load_production_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
M7_SPEC = load_prospective_spec(paths.configs / "prospective.toml")
M6_SPEC = load_production_spec(paths.configs / "production.toml")
(paths.reports / "prospective").mkdir(parents=True, exist_ok=True)
(paths.root / "artifacts" / "prospective").mkdir(parents=True, exist_ok=True)

m6_gate = paths.docs / "decisions" / "M6_GATE.md"
assert m6_gate.exists() and "M6 ✅ CLOSED" in m6_gate.read_text(encoding="utf-8"), "M6 must be CLOSED before M7."
print("M7 schema:", M7_SPEC.schema_version)
print("Protocol freeze:", M7_SPEC.core.protocol_freeze_date)
print("Prospective start:", M7_SPEC.core.prospective_start)


M7 schema: m7-prospective-v2
Protocol freeze: 2026-09-23
Prospective start: 2026-09-24


In [2]:
from argos.prospective.cohorts import (
    COHORT_IMPLEMENTATION,
    register_prospective_batches,
    shadow_span_days,
)
from argos.common.validation import add_validation
from argos.common.reports import write_report
import pandas as pd

print("implementation:", COHORT_IMPLEMENTATION)
inv = register_prospective_batches(paths.root, M7_SPEC)
registry = inv.eligible
print("eligible M7 batches:", len(registry))
print("ignored non-M7 batches:", len(inv.ignored))
registry


implementation: m7-61-cohort-registry-v2
eligible M7 batches: 0
ignored non-M7 batches: 1


,batch_file,batch_path,batch_relative_path,batch_sha256,scoring_batch_id,model_version,rows,distinct_contracts,duplicate_contract_ids,null_t0,min_t0,max_t0,future_only,batch_contract_ok,eligible_m7,eligibility_reason,cohort_month_min,cohort_month_max


In [3]:
records = []
span_days = shadow_span_days(registry)

add_validation(
    records,
    check_id="future_only_registered_batches",
    category="temporal_governance",
    passed=not registry.empty and bool(registry["future_only"].all()),
    observed={
        "registered_batches": len(registry),
        "ignored_pre_freeze_or_invalid": len(inv.ignored),
    },
    requirement="every registered M7 batch is strictly post-freeze; historical M6 replay batches are ignored, not treated as M7 evidence",
)
add_validation(
    records,
    check_id="minimum_scoring_batches",
    category="operations",
    passed=len(registry) >= M7_SPEC.cohorts.min_scoring_batches,
    observed={"eligible_batches": len(registry), "required": M7_SPEC.cohorts.min_scoring_batches},
    requirement="M7 observes multiple independent prospective scoring batches",
)
add_validation(
    records,
    check_id="minimum_shadow_span",
    category="operations",
    passed=span_days >= M7_SPEC.cohorts.min_shadow_days,
    observed={"shadow_span_days": span_days, "required": M7_SPEC.cohorts.min_shadow_days},
    requirement="prospective operation spans the preregistered minimum calendar period",
)
validation = pd.DataFrame.from_records(records)
write_report(validation, paths.reports / "prospective" / "61_validation.parquet")
blocking = validation.loc[validation["blocking"]]
ready = bool(not blocking.empty and blocking["passed"].all())
print("61 ✅ PASS" if ready else "61 ⏳ WAITING FOR PROSPECTIVE BATCHES")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("shadow span days:", span_days)
validation


61 ⏳ WAITING FOR PROSPECTIVE BATCHES
Blocking checks: 0/3
shadow span days: 0


,check_id,category,blocking,passed,observed,requirement
0,future_only_registered_batches,temporal_governance,True,False,"{""ignored_pre_freeze_or_invalid"": 1, ""register...",every registered M7 batch is strictly post-fre...
1,minimum_scoring_batches,operations,True,False,"{""eligible_batches"": 0, ""required"": 3}",M7 observes multiple independent prospective s...
2,minimum_shadow_span,operations,True,False,"{""required"": 90, ""shadow_span_days"": 0}",prospective operation spans the preregistered ...


`61` puede permanecer abierto durante semanas o meses. Eso no es un fallo técnico: es evidencia de que M7 exige observación prospectiva real en vez de reutilizar datos históricos.
